In [1]:
import polars as pl

In [2]:
df_raw = pl.read_csv(
    "app.log",
    separator="\n",
    has_header=False,
    new_columns=["raw"]
)

df_raw.head()

raw
str
"""2026-01-05 14:46:20,174 - app …"
"""2026-01-05 14:46:20,174 - app …"
"""2026-01-05 14:46:20,175 - app …"
"""2026-01-05 14:46:20,175 - app …"
"""2026-01-05 14:46:20,178 - app …"


In [3]:
df = (
    df_raw
    .with_columns(
        pl.col("raw").str.extract_groups(
            r"(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3})\s+-\s+"
            r"(?P<logger>[^ ]+)\s+-\s+"
            r"(?P<level>[A-Z]+)\s+-\s+"
            r"(?P<message>.*)"
        )
    )
    .unnest("raw")
)

df.head()

timestamp,logger,level,message
str,str,str,str
"""2026-01-05 14:46:20,174""","""app""","""INFO""","""Application started"""
"""2026-01-05 14:46:20,174""","""app""","""INFO""","""Parsing passed argumnents..."""
"""2026-01-05 14:46:20,175""","""app""","""INFO""","""Starting importing process"""
"""2026-01-05 14:46:20,175""","""app""","""INFO""","""Reading JSON file: data/locati…"
"""2026-01-05 14:46:20,178""","""app""","""INFO""","""Read 6220 records from file da…"


In [4]:
df = df.with_columns(
    pl.col("timestamp").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S,%3f"),
    pl.col("level").cast(pl.Categorical),
    pl.col("logger").cast(pl.Categorical)
)

df.head()

timestamp,logger,level,message
datetime[ms],cat,cat,str
2026-01-05 14:46:20.174,"""app""","""INFO""","""Application started"""
2026-01-05 14:46:20.174,"""app""","""INFO""","""Parsing passed argumnents..."""
2026-01-05 14:46:20.175,"""app""","""INFO""","""Starting importing process"""
2026-01-05 14:46:20.175,"""app""","""INFO""","""Reading JSON file: data/locati…"
2026-01-05 14:46:20.178,"""app""","""INFO""","""Read 6220 records from file da…"


In [5]:
df = df.filter(
    pl.any_horizontal(
        pl.col(["timestamp", "logger", "level", "message"]).is_not_null()
    )
)

df.head()

timestamp,logger,level,message
datetime[ms],cat,cat,str
2026-01-05 14:46:20.174,"""app""","""INFO""","""Application started"""
2026-01-05 14:46:20.174,"""app""","""INFO""","""Parsing passed argumnents..."""
2026-01-05 14:46:20.175,"""app""","""INFO""","""Starting importing process"""
2026-01-05 14:46:20.175,"""app""","""INFO""","""Reading JSON file: data/locati…"
2026-01-05 14:46:20.178,"""app""","""INFO""","""Read 6220 records from file da…"


In [6]:
df.group_by("level").len().sort("len", descending=True)

level,len
cat,u32
"""INFO""",66


In [7]:
anchor_time = df[1, ["timestamp"]].to_series()[0]

df.sort("timestamp").with_columns(
    (pl.col("timestamp").shift(1) - anchor_time)
    .alias("delta")
).head()

timestamp,logger,level,message,delta
datetime[ms],cat,cat,str,duration[ms]
2026-01-05 14:46:20.174,"""app""","""INFO""","""Application started""",null
2026-01-05 14:46:20.174,"""app""","""INFO""","""Parsing passed argumnents...""",0ms
2026-01-05 14:46:20.175,"""app""","""INFO""","""Starting importing process""",0ms
2026-01-05 14:46:20.175,"""app""","""INFO""","""Reading JSON file: data/locati…",1ms
2026-01-05 14:46:20.178,"""app""","""INFO""","""Read 6220 records from file da…",1ms


In [8]:
df.filter(pl.col("message").str.contains("PostgreSQL")).head()

timestamp,logger,level,message
datetime[ms],cat,cat,str
2026-01-05 14:46:20.197,"""app""","""INFO""","""Successfully connected to Post…"
2026-01-05 14:46:20.928,"""app""","""INFO""","""PostgreSQL connection closed."""
2026-01-05 14:46:20.974,"""app""","""INFO""","""Successfully connected to Post…"
2026-01-05 14:46:25.955,"""app""","""INFO""","""PostgreSQL connection closed."""
2026-01-05 14:46:25.996,"""app""","""INFO""","""Successfully connected to Post…"


In [9]:
start_time = df.select(pl.col("timestamp")).head(1).item()
end_time = df.select(pl.col("timestamp")).tail(1).item()

duration = end_time - start_time

print(f"App Started:  {start_time}")
print(f"App Finished: {end_time}")
print(f"Total Time:   {duration}")

App Started:  2026-01-05 14:46:20.174000
App Finished: 2026-01-05 14:46:30.412000
Total Time:   0:00:10.238000
